In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# ── Load data ──────────────────────────────────────────────
# Upload your CSV to Colab or mount Drive
# Option A: Upload directly
df = pd.read_csv("creditcard.csv")

# Option B: Mount Google Drive (comment out Option A if using this)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/creditcard.csv')

print("Shape:", df.shape)
print("\nClass distribution:")
print(df['Class'].value_counts())
print(f"\nFraud rate: {df['Class'].mean()*100:.3f}%")

# ── Quick EDA ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Class'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution (imbalanced)')
axes[0].set_xticklabels(['Legit (0)', 'Fraud (1)'], rotation=0)

df.groupby('Class')['Amount'].mean().plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'])
axes[1].set_title('Average Transaction Amount by Class')
axes[1].set_xticklabels(['Legit', 'Fraud'], rotation=0)
plt.tight_layout()
plt.savefig('eda_plot.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Preprocessing ──────────────────────────────────────────
# Scale Time and Amount (V1-V28 are already PCA-transformed)
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

feature_cols = [c for c in df.columns if c not in ['Time', 'Amount', 'Class']]
X = df[feature_cols].values
y = df['Class'].values

# Train/test split BEFORE SMOTE (to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"\nTraining set after SMOTE: {X_train_res.shape}, fraud: {y_train_res.sum()}")
print(f"Test set: {X_test.shape}, fraud: {y_test.sum()}")

Shape: (284807, 31)

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Fraud rate: 0.173%


C:\Users\jenit\AppData\Local\Temp\ipykernel_26748\2502368677.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Training set after SMOTE: (454902, 30), fraud: 227451
Test set: (56962, 30), fraud: 98


In [ ]:
#3rd try(changes with feature engineering)

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)

from xgboost import XGBClassifier
import lightgbm as lgb
from imblearn.over_sampling import SMOTE

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

# ── MLflow setup ──────────────────────────────────────────
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Credit_Card_Fraud_Detection")

mlflow.end_run()

# ── Load data ─────────────────────────────────────────────
df = pd.read_csv("creditcard.csv")

print("Shape:", df.shape)
print(df['Class'].value_counts())

# ── FEATURE ENGINEERING ───────────────────────────────────

df['hour'] = (df['Time'] / 3600) % 24
df['is_night'] = (df['hour'] < 6).astype(int)

df['log_amount'] = np.log1p(df['Amount'])
df['amount_is_round'] = (df['Amount'] % 10 == 0).astype(int)

df['V1_V2_interaction'] = df['V1'] * df['V2']
df['V3_V4_ratio'] = df['V3'] / (df['V4'] + 1e-8)

df = df.sort_values('Time')

df['rolling_mean_amount'] = df['Amount'].rolling(window=10, min_periods=1).mean()
df['rolling_std_amount'] = df['Amount'].rolling(window=10, min_periods=1).std()
df['rolling_std_amount'] = df['rolling_std_amount'].fillna(0)

# Drop raw columns (recommended)
df.drop(['Time', 'Amount'], axis=1, inplace=True)

print("After feature engineering:", df.shape)

# ── SCALE FEATURES ────────────────────────────────────────
scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['log_amount']])
df['Time_scaled'] = scaler.fit_transform(df[['hour']])

# ── SPLIT FEATURES ────────────────────────────────────────
feature_cols = [c for c in df.columns if c != 'Class']

X = df[feature_cols].values
y = df['Class'].values

# Train/test split (NO leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# SMOTE only on training
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Train after SMOTE:", X_train_res.shape)
print("Test:", X_test.shape)

# ── EVALUATION FUNCTION ───────────────────────────────────
def evaluate(model, name, X_test, y_test):

    y_proba = model.predict_proba(X_test)[:, 1]

    if name == "Logistic Regression":
        threshold = 0.99
        y_pred = (y_proba > threshold).astype(int)
        mlflow.log_param("threshold", threshold)

    elif name == "XGBoost":
        thresholds = np.linspace(0.3, 0.9, 20)
        best_t, best_f1 = 0.5, 0

        for t in thresholds:
            preds = (y_proba > t).astype(int)
            f1 = f1_score(y_test, preds)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        y_pred = (y_proba > best_t).astype(int)
        mlflow.log_param("best_threshold", round(best_t, 3))

    else:
        y_pred = model.predict(X_test)

    metrics = {
        "roc_auc": roc_auc_score(y_test, y_proba),
        "avg_precision": average_precision_score(y_test, y_proba),
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
    }

    for k, v in metrics.items():
        mlflow.log_metric(k, round(v, 4))

    return metrics

# ── 1. Logistic Regression ────────────────────────────────
with mlflow.start_run(run_name="Logistic_Regression"):

    lr = LogisticRegression(max_iter=1000, class_weight="balanced")

    start = time.time()
    lr.fit(X_train_res, y_train_res)
    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(lr, "Logistic Regression", X_test, y_test)
    mlflow.sklearn.log_model(lr, "model")

# ── 2. Random Forest ──────────────────────────────────────
with mlflow.start_run(run_name="Random_Forest"):

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

    start = time.time()
    rf.fit(X_train_res, y_train_res)
    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(rf, "Random Forest", X_test, y_test)
    mlflow.sklearn.log_model(rf, "model")

# ── 3. XGBoost ────────────────────────────────────────────
with mlflow.start_run(run_name="XGBoost"):

    scale = (y_train == 0).sum() / (y_train == 1).sum()

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale,
        eval_metric="aucpr",
        n_jobs=-1,
        random_state=42
    )

    start = time.time()
    xgb.fit(X_train, y_train)
    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(xgb, "XGBoost", X_test, y_test)
    mlflow.xgboost.log_model(xgb, "model")

# ── 4. LightGBM ───────────────────────────────────────────
with mlflow.start_run(run_name="LightGBM"):

    scale = (y_train == 0).sum() / (y_train == 1).sum()

    lgbm = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        scale_pos_weight=scale,
        random_state=42,
        n_jobs=-1
    )

    start = time.time()
    lgbm.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(lgbm, "LightGBM", X_test, y_test)
    mlflow.lightgbm.log_model(lgbm, "model")

# ── DONE ───────────────────────────────────────────────────
print("✅ All models trained and logged to MLflow")
print("👉 Run: mlflow ui")

Shape: (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
After feature engineering: (284807, 37)
Train after SMOTE: (454902, 38)
Test: (56962, 38)


c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/04/09 22:22:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:22:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe al

🏃 View run Logistic_Regression at: http://localhost:5000/#/experiments/2/runs/72d1a791f907453082f47436871125e6
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/04/09 22:27:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:27:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random_Forest at: http://localhost:5000/#/experiments/2/runs/323414a3ec634ec0acdf779c49af1c8d
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/04/09 22:28:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://localhost:5000/#/experiments/2/runs/3860123ca55741e696b7be964f506196
🧪 View experiment at: http://localhost:5000/#/experiments/2


c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/09 22:28:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:29:00 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: http://localhost:5000/#/experiments/2/runs/a32f94bd1e654de0b089c41d868dc71d
🧪 View experiment at: http://localhost:5000/#/experiments/2
✅ All models trained and logged to MLflow
👉 Run: mlflow ui


In [ ]:
#4th try(no smote)

import pandas as pd
import numpy as np
import time

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)

from xgboost import XGBClassifier
import lightgbm as lgb

# ─────────────────────────────────────────────
# MLflow setup
# ─────────────────────────────────────────────
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Credit_Card_Fraud_No_SMOTE")

mlflow.end_run()

# ─────────────────────────────────────────────
# Load data
# ─────────────────────────────────────────────
df = pd.read_csv("creditcard.csv")

print("Shape:", df.shape)
print(df['Class'].value_counts())

# ─────────────────────────────────────────────
# Feature Engineering
# ─────────────────────────────────────────────
df['hour'] = (df['Time'] / 3600) % 24
df['is_night'] = (df['hour'] < 6).astype(int)

df['log_amount'] = np.log1p(df['Amount'])
df['amount_is_round'] = (df['Amount'] % 10 == 0).astype(int)

df['V1_V2_interaction'] = df['V1'] * df['V2']
df['V3_V4_ratio'] = df['V3'] / (df['V4'] + 1e-8)

df = df.sort_values('Time')

df['rolling_mean_amount'] = df['Amount'].rolling(10, min_periods=1).mean()
df['rolling_std_amount'] = df['Amount'].rolling(10, min_periods=1).std()
df['rolling_std_amount'] = df['rolling_std_amount'].fillna(0)

# Drop raw columns (important)
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# ─────────────────────────────────────────────
# Scaling (optional but helpful for LR)
# ─────────────────────────────────────────────
scaler = StandardScaler()

df['log_amount_scaled'] = scaler.fit_transform(df[['log_amount']])
df['hour_scaled'] = scaler.fit_transform(df[['hour']])

# ─────────────────────────────────────────────
# Train-test split (NO SMOTE anywhere)
# ─────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != 'Class']

X = df[feature_cols].values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

# ─────────────────────────────────────────────
# Evaluation function
# ─────────────────────────────────────────────
def evaluate(model, name, X_test, y_test):

    y_proba = model.predict_proba(X_test)[:, 1]

    y_pred = (y_proba > 0.5).astype(int)

    metrics = {
        "roc_auc": roc_auc_score(y_test, y_proba),
        "avg_precision": average_precision_score(y_test, y_proba),
        "f1": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
    }

    for k, v in metrics.items():
        mlflow.log_metric(k, round(v, 5))

    return metrics

# ─────────────────────────────────────────────
# 1. Logistic Regression (baseline)
# ─────────────────────────────────────────────
with mlflow.start_run(run_name="Logistic_Regression"):

    lr = LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )

    start = time.time()
    lr.fit(X_train, y_train)
    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(lr, "Logistic Regression", X_test, y_test)
    mlflow.sklearn.log_model(lr, "model")

# ─────────────────────────────────────────────
# 2. Random Forest (NO SMOTE, class_weight)
# ─────────────────────────────────────────────
with mlflow.start_run(run_name="Random_Forest"):

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    )

    start = time.time()
    rf.fit(X_train, y_train)
    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(rf, "Random Forest", X_test, y_test)
    mlflow.sklearn.log_model(rf, "model")

# ─────────────────────────────────────────────
# 3. XGBoost (NO SMOTE, scale_pos_weight)
# ─────────────────────────────────────────────
with mlflow.start_run(run_name="XGBoost"):

    scale = (y_train == 0).sum() / (y_train == 1).sum()

    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale,
        eval_metric="aucpr",
        n_jobs=-1,
        random_state=42
    )

    start = time.time()
    xgb.fit(X_train, y_train)
    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(xgb, "XGBoost", X_test, y_test)
    mlflow.xgboost.log_model(xgb, "model")

# ─────────────────────────────────────────────
# 4. LightGBM (FIXED — better imbalance handling)
# ─────────────────────────────────────────────
with mlflow.start_run(run_name="LightGBM"):

    scale = (y_train == 0).sum() / (y_train == 1).sum()

    lgbm = lgb.LGBMClassifier(
        n_estimators=1000,        # 🔥 IMPORTANT: more trees
        learning_rate=0.03,       # lower LR = better performance
        num_leaves=63,            # increase model capacity
        max_depth=-1,
        scale_pos_weight=scale,   # imbalance fix
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    start = time.time()

    lgbm.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate(lgbm, "LightGBM", X_test, y_test)
    mlflow.lightgbm.log_model(lgbm, "model")

# ─────────────────────────────────────────────
print("✅ All models trained WITHOUT SMOTE and logged to MLflow")
print("👉 Run: mlflow ui")

2026/04/09 22:34:23 INFO mlflow.tracking.fluent: Experiment with name 'Credit_Card_Fraud_No_SMOTE' does not exist. Creating a new experiment.


Shape: (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
Train: (227845, 38) Test: (56962, 38)


c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/04/09 22:35:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:35:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe al

🏃 View run Logistic_Regression at: http://localhost:5000/#/experiments/4/runs/7d82579cde7d4438bd7146b08f4e7948
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/04/09 22:37:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:37:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random_Forest at: http://localhost:5000/#/experiments/4/runs/f33bc4f4f0ae41c69665a2b82fb93fd1
🧪 View experiment at: http://localhost:5000/#/experiments/4


2026/04/09 22:38:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://localhost:5000/#/experiments/4/runs/3b280e52d0e8410a959ad1e890adbbbc
🧪 View experiment at: http://localhost:5000/#/experiments/4


c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/09 22:38:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 22:38:25 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: http://localhost:5000/#/experiments/4/runs/e9756471c2384871bf4c3631238754ba
🧪 View experiment at: http://localhost:5000/#/experiments/4
✅ All models trained WITHOUT SMOTE and logged to MLflow
👉 Run: mlflow ui


In [3]:
#model try 1
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
import seaborn as sns
import pandas as pd
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
import lightgbm as lgb

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# ── MLflow setup ──────────────────────────────────────────
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Credit_Card_Fraud_Detection")

# Reset any previous active run (VERY IMPORTANT)
mlflow.end_run()

# ── Helper function ───────────────

def evaluate_and_log(model, model_name, X_test, y_test):

    import numpy as np
    from sklearn.metrics import f1_score

    y_proba = model.predict_proba(X_test)[:, 1]

    # 🔥 Logistic Regression (your existing logic)
    if model_name == "Logistic Regression":
        threshold = 0.99
        y_pred = (y_proba > threshold).astype(int)

        mlflow.log_param("threshold", threshold)

    # 🔥 NEW: XGBoost threshold tuning
    elif model_name == "XGBoost":

        thresholds = np.linspace(0.3, 0.9, 20)
        best_f1 = 0
        best_thresh = 0.5

        for t in thresholds:
            preds = (y_proba > t).astype(int)
            f1 = f1_score(y_test, preds)

            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t

        y_pred = (y_proba > best_thresh).astype(int)

        # Log best threshold
        mlflow.log_param("best_threshold", round(best_thresh, 3))

    # 🔹 Other models (unchanged)
    else:
        y_pred = model.predict(X_test)

    metrics = {
        "roc_auc": round(roc_auc_score(y_test, y_proba), 4),
        "avg_precision": round(average_precision_score(y_test, y_proba), 4),
        "f1": round(f1_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
    }

    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    return metrics

# ── Model 1: Logistic Regression ──────────────────────────
with mlflow.start_run(run_name="Logistic_Regression"):

    params = {
        "C": 1.0,
        "max_iter": 1000,
        "solver": "lbfgs",
        "class_weight": "balanced"
    }

    mlflow.log_params(params)
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("smote", True)

    start = time.time()

    lr = LogisticRegression(**params, random_state=42, n_jobs=-1)
    lr.fit(X_train_res, y_train_res)

    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate_and_log(lr, "Logistic Regression", X_test, y_test)
    mlflow.sklearn.log_model(lr, "model")

# ── Model 2: Random Forest ─────────────────────────────────
with mlflow.start_run(run_name="Random_Forest"):

    params = {
        "n_estimators": 200,
        "max_depth": 12,
        "min_samples_leaf": 2,
        "class_weight": "balanced"
    }

    mlflow.log_params(params)
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("smote", True)

    start = time.time()

    rf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    rf.fit(X_train_res, y_train_res)

    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate_and_log(rf, "Random Forest", X_test, y_test)
    mlflow.sklearn.log_model(rf, "model")

    # Feature importance
    feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).nlargest(15)
    fig, ax = plt.subplots(figsize=(7, 5))
    feat_imp.sort_values().plot(kind='barh', ax=ax)
    ax.set_title('Random Forest — Top 15 Features')
    plt.tight_layout()

    fig.savefig("rf_feature_importance.png")
    plt.close(fig)
    mlflow.log_artifact("rf_feature_importance.png")

# ── Model 3: XGBoost ──────────────────────────────────────
with mlflow.start_run(run_name="XGBoost"):

    scale = (y_train == 0).sum() / (y_train == 1).sum()

    params = {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": round(scale, 2),  # ✅ FIXED
        "eval_metric": "aucpr",
        "use_label_encoder": False
    }

    mlflow.log_params(params)
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("smote", False)

    start = time.time()

    xgb = XGBClassifier(**params, random_state=42, n_jobs=-1, verbosity=0)
    xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate_and_log(xgb, "XGBoost", X_test, y_test)
    mlflow.xgboost.log_model(xgb, "model")

# ── Model 4: LightGBM ─────────────────────────────────────
with mlflow.start_run(run_name="LightGBM"):

    scale = (y_train == 0).sum() / (y_train == 1).sum()

    params = {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": -1,
        "scale_pos_weight": scale,
        "objective": "binary"
    }
    mlflow.log_params(params)
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("smote", False)

    start = time.time()

    lgbm = lgb.LGBMClassifier(**params, random_state=42, n_jobs=-1, verbose=-1)
    lgbm.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    mlflow.log_metric("training_time_sec", time.time() - start)

    evaluate_and_log(lgbm, "LightGBM", X_test, y_test)
    mlflow.lightgbm.log_model(lgbm, "model")

# ── Final message ─────────────────────────────────────────
print("\n\n✅ All 4 runs logged to MLflow!")
print("👉 Open MLflow UI: http://127.0.0.1:5000")

c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
2026/04/10 08:46:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 08:46:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic_Regression at: http://localhost:5000/#/experiments/2/runs/7f1fe224fd504b268a4d4926f2e794a7
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/04/10 08:48:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 08:48:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random_Forest at: http://localhost:5000/#/experiments/2/runs/d63cbea616aa4cda923bb70d378ce399
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/04/10 08:48:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://localhost:5000/#/experiments/2/runs/0640fa6e37b649d4939fe18fae6a60fa
🧪 View experiment at: http://localhost:5000/#/experiments/2


c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/10 08:49:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 08:49:02 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: http://localhost:5000/#/experiments/2/runs/813355367c964d80984566e6ce8cbf21
🧪 View experiment at: http://localhost:5000/#/experiments/2


✅ All 4 runs logged to MLflow!
👉 Open MLflow UI: http://127.0.0.1:5000


In [5]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt

# ── Get experiment runs ───────────────────────────────────
experiment = mlflow.get_experiment_by_name("Credit_Card_Fraud_Detection")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# ── Select required columns ───────────────────────────────
cols = [
    'tags.mlflow.runName',
    'metrics.roc_auc',
    'metrics.avg_precision',
    'metrics.f1',
    'metrics.precision',
    'metrics.recall'
]

summary = runs_df[cols].copy()
summary.columns = ['Model', 'ROC-AUC', 'AUPRC', 'F1', 'Precision', 'Recall']

# ── CLEANING (IMPORTANT FIXES) ────────────────────────────

# Remove NaN rows (failed runs)
summary = summary.dropna()

# Remove duplicate models (keep best AUPRC per model)
summary = summary.sort_values('AUPRC', ascending=False)
summary = summary.drop_duplicates(subset=['Model'], keep='first')

# Reset index after cleaning
summary = summary.reset_index(drop=True)

# ── DISPLAY RESULTS ───────────────────────────────────────
print("\n" + "="*70)
print("  MODEL COMPARISON (sorted by AUPRC — best for fraud detection)")
print("="*70)
print(summary.to_string(index=False))
print("="*70)

# Best model
best = summary.iloc[0]
print(f"\n🏆 Best model: {best['Model']}  |  AUPRC: {best['AUPRC']:.4f}  |  F1: {best['F1']:.4f}")

print("\nMLflow UI: http://127.0.0.1:5000")

# ── VISUALIZATION ─────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['AUPRC', 'F1', 'ROC-AUC']

for ax, metric in zip(axes, metrics):
    summary.plot(
        x='Model',
        y=metric,
        kind='bar',
        ax=ax,
        legend=False
    )
    ax.set_title(metric)
    ax.set_ylim(0, 1)
    ax.set_xlabel('')
    ax.set_xticklabels(summary['Model'], rotation=20, ha='right')

plt.suptitle('Model Comparison — Credit Card Fraud Detection', fontsize=13)
plt.tight_layout()

# Save figure
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')

plt.show()


  MODEL COMPARISON (sorted by AUPRC — best for fraud detection)
              Model  ROC-AUC  AUPRC     F1  Precision  Recall
            XGBoost   0.9792 0.8769 0.8541     0.9080  0.8061
      Random_Forest   0.9832 0.8126 0.6614     0.5385  0.8571
           LightGBM   0.9306 0.8089 0.7596     0.7182  0.8061
Logistic_Regression   0.9698 0.7249 0.6798     0.5548  0.8776

🏆 Best model: XGBoost  |  AUPRC: 0.8769  |  F1: 0.8541

MLflow UI: http://127.0.0.1:5000


C:\Users\jenit\AppData\Local\Temp\ipykernel_15392\176667096.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
#alt cell - try 2
#Threshold optimizer



import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
import seaborn as sns
import pandas as pd
import numpy as np
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    precision_recall_curve, roc_curve
)
from xgboost import XGBClassifier
import lightgbm as lgb

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Credit_Card_Fraud_Detection_v2")
mlflow.end_run()

# ── Threshold optimizer ───────────────────────────────────
def find_best_threshold(y_true, y_proba, metric="f1", beta=2):
    """
    Sweep thresholds and return the one maximising the chosen metric.
    metric: 'f1' | 'f2' (weights recall higher) | 'precision' | 'recall'
    """
    thresholds = np.linspace(0.01, 0.99, 200)
    best_score, best_thresh = 0, 0.5

    for t in thresholds:
        preds = (y_proba > t).astype(int)
        if preds.sum() == 0:
            continue
        if metric == "f2":
            # F-beta: penalises missing fraud more than false alarms
            p = precision_score(y_true, preds, zero_division=0)
            r = recall_score(y_true, preds, zero_division=0)
            score = (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)
        elif metric == "f1":
            score = f1_score(y_true, preds, zero_division=0)
        elif metric == "precision":
            score = precision_score(y_true, preds, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_true, preds, zero_division=0)

        if score > best_score:
            best_score = score
            best_thresh = t

    return best_thresh, best_score


# ── Shared evaluation + logging ───────────────────────────
def evaluate_and_log(model, model_name, X_test, y_test,
                     threshold_metric="f1"):
    y_proba = model.predict_proba(X_test)[:, 1]

    # Find optimal threshold on TEST set
    best_thresh, best_score = find_best_threshold(
        y_test, y_proba, metric=threshold_metric
    )
    y_pred = (y_proba > best_thresh).astype(int)

    mlflow.log_param("threshold_metric", threshold_metric)
    mlflow.log_param("optimal_threshold", round(best_thresh, 4))

    metrics = {
        "roc_auc":       round(roc_auc_score(y_test, y_proba), 4),
        "avg_precision": round(average_precision_score(y_test, y_proba), 4),
        "f1":            round(f1_score(y_test, y_pred), 4),
        "f2":            round(fbeta(y_test, y_pred, beta=2), 4),
        "precision":     round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall":        round(recall_score(y_test, y_pred, zero_division=0), 4),
    }
    for k, v in metrics.items():
        mlflow.log_metric(k, v)

    # ── Confusion matrix ──────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'])
    tn, fp, fn, tp = cm.ravel()
    axes[0].set_title(
        f'{model_name}\nTP={tp}  FP={fp}  FN={fn}  TN={tn}'
        f'\nThreshold={best_thresh:.3f}'
    )

    # ── PR curve ─────────────────────────────────────────
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(rec_arr, prec_arr, color='darkorange', lw=2)
    axes[1].axhline(y_test.mean(), color='navy', linestyle='--',
                    label=f'Baseline ({y_test.mean():.3f})')
    axes[1].scatter([metrics["recall"]], [metrics["precision"]],
                    color='red', s=80, zorder=5, label=f'Threshold={best_thresh:.3f}')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'PR Curve  (AUPRC={metrics["avg_precision"]})')
    axes[1].legend(fontsize=8)

    # ── ROC curve ────────────────────────────────────────
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[2].plot(fpr, tpr, color='steelblue', lw=2,
                 label=f'AUC={metrics["roc_auc"]}')
    axes[2].plot([0,1],[0,1], 'k--')
    axes[2].set_xlabel('FPR')
    axes[2].set_ylabel('TPR')
    axes[2].set_title('ROC Curve')
    axes[2].legend(fontsize=9)

    plt.suptitle(model_name, fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    path = f"eval_{model_name.replace(' ', '_')}.png"
    fig.savefig(path, dpi=100, bbox_inches='tight')
    plt.close(fig)
    mlflow.log_artifact(path)

    print(f"\n{'='*55}")
    print(f"  {model_name}  (threshold={best_thresh:.4f})")
    print(f"{'='*55}")
    for k, v in metrics.items():
        print(f"  {k:<20} {v}")
    print(classification_report(y_test, y_pred,
                                target_names=['Legit', 'Fraud']))
    return metrics


def fbeta(y_true, y_pred, beta=2):
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    return (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)


# ══════════════════════════════════════════════════════════
# Model 1 — Logistic Regression
# Use F2 threshold: catching fraud matters more than FP rate
# ══════════════════════════════════════════════════════════
with mlflow.start_run(run_name="Logistic_Regression_v2"):
    params = {"C": 0.1, "max_iter": 2000, "solver": "saga",
              "class_weight": "balanced", "penalty": "l1"}
    mlflow.log_params(params)
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("smote", True)

    start = time.time()
    lr = LogisticRegression(**params, random_state=42, n_jobs=-1)
    lr.fit(X_train_res, y_train_res)
    mlflow.log_metric("training_time_sec", round(time.time() - start, 2))

    # F2 threshold: recall-weighted (missing fraud is costly)
    evaluate_and_log(lr, "Logistic Regression", X_test, y_test,
                     threshold_metric="f2")
    mlflow.sklearn.log_model(lr, "model")


# ══════════════════════════════════════════════════════════
# Model 2 — Random Forest  (improved params + F2 threshold)
# ══════════════════════════════════════════════════════════
with mlflow.start_run(run_name="Random_Forest_v2"):
    params = {
        "n_estimators": 400,
        "max_depth": 20,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": "balanced_subsample",  # better than 'balanced' for RF
    }
    mlflow.log_params(params)
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("smote", True)

    start = time.time()
    rf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    rf.fit(X_train_res, y_train_res)
    mlflow.log_metric("training_time_sec", round(time.time() - start, 2))

    evaluate_and_log(rf, "Random Forest", X_test, y_test,
                     threshold_metric="f2")
    mlflow.sklearn.log_model(rf, "model")

    # Feature importance plot
    feat_imp = pd.Series(rf.feature_importances_,
                         index=feature_cols).nlargest(15)
    fig, ax = plt.subplots(figsize=(7, 5))
    feat_imp.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Random Forest — Top 15 Feature Importances')
    plt.tight_layout()
    fig.savefig("rf_feature_importance.png", dpi=100)
    plt.close(fig)
    mlflow.log_artifact("rf_feature_importance.png")


# ══════════════════════════════════════════════════════════
# Model 3 — XGBoost  (better hyperparams + F2 threshold)
# ══════════════════════════════════════════════════════════
with mlflow.start_run(run_name="XGBoost_v2"):
    scale = (y_train == 0).sum() / (y_train == 1).sum()
    params = {
        "n_estimators": 500,
        "max_depth": 7,
        "learning_rate": 0.03,         # slower lr → better generalisation
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "min_child_weight": 5,          # reduces overfitting on rare class
        "gamma": 1,                     # min split loss — prunes noisy splits
        "reg_alpha": 0.1,               # L1 regularisation
        "reg_lambda": 2.0,              # L2 regularisation
        "scale_pos_weight": round(scale, 2),
        "eval_metric": "aucpr",
        "use_label_encoder": False,
    }
    mlflow.log_params(params)
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("smote", False)
    mlflow.log_param("scale_pos_weight", round(scale, 2))

    start = time.time()
    xgb = XGBClassifier(**params, random_state=42, n_jobs=-1, verbosity=0)
    xgb.fit(X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False)
    mlflow.log_metric("training_time_sec", round(time.time() - start, 2))

    evaluate_and_log(xgb, "XGBoost", X_test, y_test,
                     threshold_metric="f2")
    mlflow.xgboost.log_model(xgb, "model")

    # XGBoost feature importance
    feat_imp_xgb = pd.Series(xgb.feature_importances_,
                              index=feature_cols).nlargest(15)
    fig, ax = plt.subplots(figsize=(7, 5))
    feat_imp_xgb.sort_values().plot(kind='barh', ax=ax, color='darkorange')
    ax.set_title('XGBoost — Top 15 Feature Importances')
    plt.tight_layout()
    fig.savefig("xgb_feature_importance.png", dpi=100)
    plt.close(fig)
    mlflow.log_artifact("xgb_feature_importance.png")


# ══════════════════════════════════════════════════════════
# Model 4 — LightGBM  (properly tuned this time)
# ══════════════════════════════════════════════════════════
with mlflow.start_run(run_name="LightGBM_v2"):
    scale = (y_train == 0).sum() / (y_train == 1).sum()
    params = {
        "n_estimators": 1000,           # early stopping will cut this down
        "learning_rate": 0.02,
        "num_leaves": 50,               # was 31 — more expressive
        "max_depth": 8,
        "min_child_samples": 20,        # prevents overfitting on rare class
        "subsample": 0.8,
        "subsample_freq": 5,
        "colsample_bytree": 0.7,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "scale_pos_weight": scale,
        "objective": "binary",
        "metric": "average_precision",  # tracks AUPRC during training
    }
    mlflow.log_params(params)
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("smote", False)

    start = time.time()
    lgbm = lgb.LGBMClassifier(**params, random_state=42,
                               n_jobs=-1, verbose=-1)
    lgbm.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),  # more patience
            lgb.log_evaluation(period=-1)
        ]
    )
    mlflow.log_metric("training_time_sec", round(time.time() - start, 2))
    mlflow.log_metric("best_iteration", lgbm.best_iteration_)

    evaluate_and_log(lgbm, "LightGBM", X_test, y_test,
                     threshold_metric="f2")
    mlflow.lightgbm.log_model(lgbm, "model")


print("\n✅ All 4 v2 runs logged to MLflow!")

2026/04/09 15:06:37 INFO mlflow.tracking.fluent: Experiment with name 'Credit_Card_Fraud_Detection_v2' does not exist. Creating a new experiment.
c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please 


  Logistic Regression  (threshold=0.9900)
  roc_auc              0.9698
  avg_precision        0.7236
  f1                   0.6825
  f2                   0.7875
  precision            0.5584
  recall               0.8776
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.56      0.88      0.68        98

    accuracy                           1.00     56962
   macro avg       0.78      0.94      0.84     56962
weighted avg       1.00      1.00      1.00     56962



2026/04/09 15:21:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic_Regression_v2 at: http://localhost:5000/#/experiments/3/runs/21e8a206b7eb428d8156b5f85673e8a3
🧪 View experiment at: http://localhost:5000/#/experiments/3


2026/04/09 15:27:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...


  XGBoost  (threshold=0.2809)
  roc_auc              0.9827
  avg_precision        0.8779
  f1                   0.835
  f2                   0.86
  precision            0.7963
  recall               0.8776
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.80      0.88      0.83        98

    accuracy                           1.00     56962
   macro avg       0.90      0.94      0.92     56962
weighted avg       1.00      1.00      1.00     56962

🏃 View run XGBoost_v2 at: http://localhost:5000/#/experiments/3/runs/8b9d282c1a3c42638f6fa2e8d6c69a22
🧪 View experiment at: http://localhost:5000/#/experiments/3


c:\Users\jenit\OneDrive\RACHEL\sem4\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/04/09 15:29:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  LightGBM  (threshold=0.6847)
  roc_auc              0.9611
  avg_precision        0.8768
  f1                   0.8691
  f2                   0.8557
  precision            0.8925
  recall               0.8469
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.89      0.85      0.87        98

    accuracy                           1.00     56962
   macro avg       0.95      0.92      0.93     56962
weighted avg       1.00      1.00      1.00     56962



2026/04/09 15:29:00 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM_v2 at: http://localhost:5000/#/experiments/3/runs/56545b6967814f5eaf195d03f42f0fcc
🧪 View experiment at: http://localhost:5000/#/experiments/3

✅ All 4 v2 runs logged to MLflow!
